In [ ]:
import pandas as pd

file_path = "total_data.csv"
df = pd.read_csv(file_path)

# '주차' 컬럼을 날짜 형식으로 변환
df['주차'] = pd.to_datetime(df['주차'], format='%Y%m%d')


# 특정 라인(ZD, ZE, ZF) 제거
df = df[~df['라인'].isin(['ZD', 'ZE', 'ZF'])]
# '소품', '언더웨어' 제거
df = df[~df['복종'].isin(['소품', '언더웨어'])]


# '복종'과 '소품종' 컬럼을 결합하여 새로운 '카테고리' 컬럼 생성
df['카테고리'] = df['복종'] + "_" + df['소품종'] + "_" + df['라인']


# 실판매가 계산 (판매액 / 판매수량)
df['실판매가'] = df['판매액'] / df['판매수량']

# 택가 계산 (판매택가 / 판매수량)
df['택가'] = df['판매택가'] / df['판매수량']

# 할인율 계산 ((택가 - 실판매가) / 택가)
df['할인율'] = (df['택가'] - df['실판매가']) * 100 / df['택가']

# 그룹화 및 집계 연산 적용
group_cols = ['주차', '시즌', '카테고리']
agg_dict = {
    '할인율': 'mean',
    
    '판매액': 'sum',
    '판매수량': 'sum',
    '매출원가': 'sum',
    '판매택가': 'sum',
    
    '실판매가' : 'mean',
    '택가' : 'mean'
}

df = df.groupby(group_cols).agg(agg_dict).reset_index()

# NaN 값이 있는 경우 0으로 채우기
df.fillna(0, inplace=True)

df

,주차,시즌,카테고리,할인율,판매액,판매수량,매출원가,판매택가,실판매가,택가
0,2021-01-03,봄,가죽&FUR_가죽점퍼_ZA,1.130573,52297400,53,10846000,52947000,987705.572755,999000.0
1,2021-01-03,봄,니트 셔츠_라운드_ZB,32.281514,18981424,401,3559559,28029900,47335.221945,69900.0
2,2021-01-03,봄,수트_블레이져(수트)_ZA,39.444496,19685400,109,5717811,35131000,207972.059315,339000.0
3,2021-01-03,봄,수트_블레이져(수트)_ZB,-0.265138,29890941,194,7913293,38606000,199527.625451,199000.0
4,2021-01-03,봄,수트_블레이져(수트)_ZC,22.736561,5652700,33,1610764,8727000,175803.703704,239000.0
...,...,...,...,...,...,...,...,...,...,...
12265,2024-12-29,여름,팬츠_반바지_ZB,0.000000,0,0,0,0,0.000000,0.0
12266,2024-12-29,여름,팬츠_반바지_ZG,0.000000,0,0,0,0,0.000000,0.0
12267,2024-12-29,여름,팬츠_팬츠(일반)_ZA,0.000000,0,0,0,0,0.000000,0.0
12268,2024-12-29,여름,팬츠_팬츠(일반)_ZB,0.000000,0,0,0,0,0.000000,0.0


In [4]:
import numpy as np

# 판매수량이 음수인 값만 필터링
negative_sales_df = df[df['판매수량'] < 0]

# IQR 계산
Q1 = negative_sales_df['판매수량'].quantile(0.25)
Q3 = negative_sales_df['판매수량'].quantile(0.75)
IQR = Q3 - Q1

# 3 * IQR 기준으로 이상치 판별
lower_bound = Q1 - 3 * IQR

# 이상치 개수 확인
outliers_count = (negative_sales_df['판매수량'] < lower_bound).sum()

# 결과 출력
print(f"이상치 기점: {lower_bound}, 3 IQR 이상인 이상치 개수: {outliers_count}")


이상치 기점: -139.0, 3 IQR 이상인 이상치 개수: 40


In [5]:
# 카테고리별 개수 세기

category_counts = negative_sales_df[negative_sales_df['판매수량'] < lower_bound]
category_outlier_counts = category_counts['카테고리'].value_counts()
category_outlier_counts

니트 셔츠_라운드_ZB            13
점퍼_패딩점퍼_ZB               6
데님_데님팬츠_ZB               6
자켓_싱글재킷_ZB               3
스웨터_라운드_ZB               2
스웨터_T-에리_ZB              2
팬츠_반바지_ZB                1
수트_블레이져(수트)_ZB           1
스웨터_오픈형(CARDIGAN)_ZB     1
조끼_패딩베스트_ZB              1
수트_수트팬츠_ZB               1
우븐 셔츠_캐쥬얼셔츠_ZA           1
코트_싱글코트_ZB               1
우븐 셔츠_드레스셔츠_ZB           1
Name: 카테고리, dtype: int64

In [6]:
def update_data(df, input_rows, output_rows):
    df = df.copy()  # 원본 데이터 보호

    # output 행 값 변경 (input과 output을 합쳐서 덮어쓰기)
    for i in range(len(input_rows)):
        input_date, season, category = input_rows[i]
        output_date, _, _ = output_rows[i]

        # input과 output 행 찾기
        mask_input = (df['주차'] == input_date) & (df['시즌'] == season) & (df['카테고리'] == category)
        mask_output = (df['주차'] == output_date) & (df['시즌'] == season) & (df['카테고리'] == category)

        if mask_input.sum() == 0 or mask_output.sum() == 0:
            print(f"해당하는 input/output 데이터 없음: {input_rows[i]} → {output_rows[i]}")
            continue

        # input 행과 output 행 값 합치기
        df.loc[mask_output, ['판매액', '판매수량', '매출원가', '판매택가']] = (
            df.loc[mask_input, ['판매액', '판매수량', '매출원가', '판매택가']].values +
            df.loc[mask_output, ['판매액', '판매수량', '매출원가', '판매택가']].values
        )

        # 실판매가, 택가, 할인율 다시 계산
        df.loc[mask_output, '실판매가'] = df.loc[mask_output, '판매액'] / df.loc[mask_output, '판매수량']
        df.loc[mask_output, '택가'] = df.loc[mask_output, '판매택가'] / df.loc[mask_output, '판매수량']
        df.loc[mask_output, '할인율'] = (df.loc[mask_output, '판매택가'] - df.loc[mask_output, '판매액']) * 100 / df.loc[mask_output, '판매택가']

    # input 행 값 변경 (이전, 이후 주차 평균으로 대체)
    for input_date, season, category in input_rows:
        mask_input = (df['주차'] == input_date) & (df['시즌'] == season) & (df['카테고리'] == category)

        if mask_input.sum() == 0:
            print(f"해당하는 input 데이터 없음: {input_date}, {season}, {category}")
            continue

        # 동일한 시즌, 카테고리에서 이전/이후 주차 찾기
        prev_week = df[(df['주차'] < input_date) & (df['시즌'] == season) & (df['카테고리'] == category)].sort_values('주차', ascending=False).head(1)
        next_week = df[(df['주차'] > input_date) & (df['시즌'] == season) & (df['카테고리'] == category)].sort_values('주차', ascending=True).head(1)

        # 이전 & 이후 주차 평균값 구하기
        if not prev_week.empty and not next_week.empty:
            mean_values = (prev_week[['판매액', '판매수량', '매출원가', '판매택가','할인율']].values + next_week[['판매액', '판매수량', '매출원가', '판매택가','할인율']].values) / 2
        elif not prev_week.empty:
            mean_values = prev_week[['판매액', '판매수량', '매출원가', '판매택가','할인율']].values
        elif not next_week.empty:
            mean_values = next_week[['판매액', '판매수량', '매출원가', '판매택가','할인율']].values
        else:
            print(f"이전/이후 주차 데이터 없음: {input_date}, {season}, {category}")
            continue

        # input 행 값 변경
        df.loc[mask_input, ['판매액', '판매수량', '매출원가', '판매택가','할인율']] = mean_values

        # 실판매가, 택가, 할인율 다시 계산
        df.loc[mask_input, '실판매가'] = df.loc[mask_input, '판매액'] / df.loc[mask_input, '판매수량']
        df.loc[mask_input, '택가'] = df.loc[mask_input, '판매택가'] / df.loc[mask_input, '판매수량']

        df.fillna(0, inplace=True)  # NaN 값 0으로 채우기

    return df

# 실행 예시 (여기에 원하는 데이터 콤마로 이어서 쭉쭉 넣으면 됨)
input_rows = [
    ('2023-07-09', '사계절', '니트 셔츠_라운드_ZB')
]
output_rows = [
    ('2023-06-18', '사계절', '니트 셔츠_라운드_ZB')
]

df_updated = update_data(df, input_rows, output_rows)

# 결과 확인
df_updated.loc[
    (df_updated['주차'].isin(['2023-06-18', '2023-07-09'])) & 
    (df_updated['카테고리'] == '니트 셔츠_라운드_ZB')
]



,주차,시즌,카테고리,할인율,판매액,판매수량,매출원가,판매택가,실판매가,택가
6958,2023-06-18,사계절,니트 셔츠_라운드_ZB,63.869037,9757600.0,338.0,2511267.0,27006200.0,2.886864e+04,79900.0
6969,2023-06-18,여름,니트 셔츠_라운드_ZB,65.306768,123119373.0,5373.0,35765711.0,353302700.0,2.588424e+04,74900.0
7111,2023-07-09,사계절,니트 셔츠_라운드_ZB,60.956967,-1352150.0,0.5,3715.0,39950.0,-2.704300e+06,79900.0
7122,2023-07-09,여름,니트 셔츠_라운드_ZB,68.658115,89118467.0,4620.0,29557538.0,297718000.0,2.371387e+04,74900.0
